# Dynamic Augmentation Workflow

This workflow uses a **fixed, range-based data augmentation policy** combined with a **dynamic heatmap target difficulty**.  
All input augmentations remain constant throughout training, while the **Gaussian heatmap sigma (σ)** is gradually reduced across epochs to guide learning from coarse localization to precise landmark detection.

---

### One Scan = One Sample
Each training sample corresponds to a single MRI scan and contains all vertebra levels as multi-channel heatmaps.  
This preserves anatomical context and prevents data leakage.

---

### Fixed Augmentation Policy
A single augmentation policy is defined once and reused for all epochs.
Augmentations are applied **on-the-fly** with random intensities sampled from predefined ranges.

- Dataset remains unchanged across epochs
- Augmentation variability comes from random sampling
- No curriculum on augmentation strength

---

### Range-Based Random Augmentations

**Geometric (image + heatmaps):**
- Rotation (±θ)
- Translation (fraction of image size)
- Scaling
- Horizontal flip (probabilistic)
- Vertical flip (low probability, optional)

**Intensity (image only):**
- Contrast adjustment
- Gamma correction
- Gaussian noise
- Strong Gaussian blur (fixed strength, probabilistic)

---

### Blur for Coarse Structure Learning
Gaussian blur suppresses fine image details and texture cues, encouraging the model to learn global anatomical structure.
Blur strength and probability are fixed throughout training.

---

### Dynamic Heatmap Sigma (σ)
The only dynamic element in training is the Gaussian heatmap standard deviation:

- Early training: large σ → easy, coarse targets
- Late training: small σ → precise localization

σ is updated once per epoch via `set_epoch(epoch)`.

## Data augmentation

In [2]:
print("Tralala")

In [2]:
import os, glob
import numpy as np
import pandas as pd
import zipfile
from pathlib import Path
import matplotlib.pyplot as plt
from tensorflow.keras.preprocessing import image
import cv2
import re
from PIL import Image
import torch
from torch.utils.data import Dataset
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms
from tqdm import tqdm
from torch.utils.data import random_split, DataLoader
import math
import torch
import torch.nn as nn

import random
import torch
import torchvision.transforms.functional as TF
from torchvision.transforms import InterpolationMode

import torch.nn.functional as F

In [1]:
os. chdir("/home/jupyter-lukj08@vse.cz/BC/Pre-training")

In [5]:
%run ./Pretraining-fce.ipynb

In [8]:
%run ./Pretraining-models.ipynb

In [9]:
%run ./Pretraining-datasets.ipynb

In [10]:
BASE_DIR = Path.cwd()
print(BASE_DIR)

In [6]:
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"       # quieter logs (optional)
os.environ["TF_FORCE_GPU_ALLOW_GROWTH"] = "true"  # enables growth mode
torch.cuda.is_available(), torch.cuda.get_device_name(0)
device = "cuda" if torch.cuda.is_available() else "cpu"

In [7]:
df_coords = pd.read_csv("df_coords.csv")
print("Loaded df_coords:")
display(df_coords)

In [8]:
path = './data/processed_osf_jpgs/ID39.jpg'
img = Image.open(path)
print(img.size) 

# Normalize for training
IMG_SIZE = 256
mean = [0.2127]
std = [0.2673]

## Dataset creation & augmentation

In [9]:
# ---- Split (no leakage) ----
df_train, df_val, df_test = split_by_scan(df_coords, val_frac=0.2, seed=42)

In [18]:
num_epochs = 30
schedule = CurriculumSchedule(
    num_epochs=num_epochs,
    blur_p_start=0.5,
    blur_decay_portion=0.35,
    sigma_start=10.0,
    sigma_end=3.0
)

train_ds = SpineLandmarksDatasetDynamAug(df_train, img_size=256, schedule=schedule)
val_ds   = SpineLandmarksDatasetDynamAug(df_val,img_size=256, schedule=None)
test_ds   = SpineLandmarksDatasetDynamAug(df_test,img_size=256, schedule=None)

print(f"Train scans: {len(train_ds)}, Val scans: {len(val_ds)}, Test scans: {len(val_ds)}")


In [15]:
unaug_train_ds = SpineLandmarksDatasetDynamAug(df_train, img_size=256, schedule=None)
# train_ds is your augmented dataset with schedule

idx = 4  # try a few indices
show_sample_pair(unaug_train_ds, train_ds, idx, epoch=0)
show_sample_pair(unaug_train_ds, train_ds, idx, epoch=num_epochs//2)
show_sample_pair(unaug_train_ds, train_ds, idx, epoch=num_epochs-1)

In [19]:
# DataLoaders
BATCH_SIZE = 8
NUM_WORKERS = 2

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=True, drop_last=False
)

val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True, drop_last=False
)

test_loader = DataLoader(
    test_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True, drop_last=False
)

print("Train batches:", len(train_loader), "Val batches:", len(val_loader))
print("Levels:", train_ds.levels)

## Models trainig

In [24]:
# CONFIG

class WeightedMSELoss(nn.Module):
    def __init__(self, alpha=10.0):
        super().__init__()
        self.alpha = float(alpha)

    def forward(self, pred, target):
        # pred/target: [B,K,H,W]
        w = 1.0 + self.alpha * target  # emphasize landmark area
        return (w * (pred - target) ** 2).mean()

use_amp = torch.cuda.is_available()
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
patience = 10
criterion = WeightedMSELoss(alpha=10.0)


### U-Net

In [21]:
model1 = UNet(
    in_channels=1,   # grayscale
    n_classes=5,
).to(device)

optimizer1 = torch.optim.Adam(model1.parameters(), lr=1e-3, weight_decay=1e-4)

print("Model ready on", device)

total_params = sum(p.numel() for p in model1.parameters())
print(f"Total parameters: {total_params:,}")

In [25]:
model1, hist1 = train_with_curriculum(
    model=model1,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer1,
    device=device,
    criterion=criterion,
    num_epochs=num_epochs,
    patience=patience,
    scaler=scaler,
    use_amp=use_amp,
)

### Res U-Net

In [26]:
model2 = ResUNet(
    in_channels=1,   # grayscale
    n_classes=5,
    base_c=64        # matches the diagram; you can use 32 if you want smaller model
).to(device)

optimizer2 = torch.optim.Adam(model2.parameters(), lr=1e-3, weight_decay=1e-4)

print("Model ready on", device)

total_params = sum(p.numel() for p in model2.parameters())
print(f"Total parameters: {total_params:,}")

In [27]:
model2, hist2 = train_with_curriculum(
    model=model2,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer2,
    device=device,
    criterion=criterion,
    num_epochs=num_epochs,
    patience=patience,
    scaler=scaler,
    use_amp=use_amp,
)

### Attention Res-U-Net

In [28]:
model3 = AttentionResUNet(
    in_channels=1,   # grayscale
    n_classes=5,
    base_c=64  
).to(device)

optimizer3 = torch.optim.Adam(model3.parameters(), lr=1e-3, weight_decay=1e-4)

print("Model ready on", device)

total_params = sum(p.numel() for p in model3.parameters())
print(f"Total parameters: {total_params:,}")

In [30]:
model3, hist3 = train_with_curriculum(
    model=model3,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer3,
    device=device,
    criterion=criterion,
    num_epochs=num_epochs,
    patience=patience,
    scaler=scaler,
    use_amp=use_amp,
)

## Models Evaluation

In [31]:
# Validation
table_val, rows_val = compare_models_pixel_error_heatmap(
    models=[model1, model2, model3],
    model_names=["UNet", "Res-UNet", "Attention-Res-UNet"],
    loader=val_loader,
    device=device,
    use_amp=True,
)

In [32]:
# Test
table_test, rows_test = compare_models_pixel_error_heatmap(
    models=[model1, model2, model3],
    model_names=["UNet", "Res-UNet", "Attention-Res-UNet"],
    loader=test_loader,
    device=device,
    use_amp=True,
)

## Models Evaluation
The primary evaluation metric is **pixel localization error**:

- Euclidean distance (in pixels) between:
  - Ground truth heatmap argmax
  - Predicted heatmap argmax

Reported metrics include:
- Per-level mean pixel error
- Overall mean pixel error
- Overall 90th percentile (p90) pixel error


### Architectural Comparison
- **Res-U-Net > U-Net**: residual connections improve both accuracy and robustness.
- **Attention Res-U-Net** does not improve performance and slightly degrades results, suggesting attention adds complexity without clear benefit for this task and dataset size.

---

### Conclusion
- **Res-U-Net** is the best-performing and most stable architecture for vertebra localization in this setup.
- Remaining errors are dominated by difficult upper-spine cases and occasional large failures (high p90), indicating room for improvement in handling full-spine images and level ambiguity.

In [33]:
# Validation
table_val, rows_val = compare_models_pixel_error_heatmap(
    models=[model1, model2, model3],
    model_names=["UNet", "Res-UNet", "Attention-Res-UNet"],
    loader=val_loader,
    device=device,
    use_amp=True,
)

In [34]:
# Test
table_test, rows_test = compare_models_pixel_error_heatmap(
    models=[model1, model2, model3],
    model_names=["UNet", "Res-UNet", "Attention-Res-UNet"],
    loader=test_loader,
    device=device,
    use_amp=True,
)

### Best models performance evaluation

In [40]:
mean = [0.2127]
std  = [0.2673]

mean_t = torch.tensor(mean, dtype=torch.float32)
std_t  = torch.tensor(std,  dtype=torch.float32)

for idx in range(10):
    print(f"\nVisualizing val sample {idx}/{len(val_ds)-1}")

    visualize_sample(
        model=model3,
        dataset=val_ds,
        idx=idx,
        device=device,
        mean=mean_t,
        std=std_t,
    )